# Case Study 1 — EXP-1-IDABS Random Forest

In [ ]:

REPRESENTATION_MODE = "idabs"
CODE_COLUMN = "abstracted_code_v1"

RUN_PROFILE_FOLD = False
PROFILE_FOLD_ID = 0

RUN_OFFICIAL_DEV_CV = True

RUN_VERSION = "v1"

ALLOW_OVERWRITE_OUTPUT_DIR = False

RUN_FINAL_OUTER_HOLDOUT = True

REBUILD_STATIC_FEATURE_CACHE = False

REPO_URL = "https://github.com/EnomisLP/DiverseVul--IS-Project.git"
REPO_BRANCH = "prashant"

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
except Exception as exc:
    print("Google Drive mount skipped or unavailable:", exc)

REPO_ROOT = Path("/content/DiverseVul--IS-Project")
PROJECT_DIR = REPO_ROOT / "vuln-detection"
SRC_DIR = PROJECT_DIR / "src"

if not REPO_ROOT.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "origin"], check=False)
    subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH], check=False)
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "origin", REPO_BRANCH], check=False)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Repository:", REPO_ROOT)
print("Project dir:", PROJECT_DIR)
print("Source dir:", SRC_DIR)
print("Python path contains source dir:", str(SRC_DIR) in sys.path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repository: /content/DiverseVul--IS-Project
Project dir: /content/DiverseVul--IS-Project/vuln-detection
Source dir: /content/DiverseVul--IS-Project/vuln-detection/src
Python path contains source dir: True


In [ ]:
import importlib
import json
import shutil
import time

packages = [
    "numpy",
    "pandas",
    "scipy",
    "scikit-learn",
    "pyarrow",
    "matplotlib",
    "joblib",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *packages],
    check=True,
)

import numpy as np
import pandas as pd
from IPython.display import display


try:
    exp1_rf = importlib.import_module("case_study_1.exp1.exp1_rf")
    static_features = importlib.import_module("case_study_1.exp1.static_features")
except ModuleNotFoundError:
    exp1_rf = importlib.import_module("case_study_1.exp1_rf")
    static_features = importlib.import_module("case_study_1.static_features")

split_manifest = importlib.import_module("case_study_1.split_manifest")
evaluation = importlib.import_module("case_study_1.evaluation")

required_exp1_api = [
    "EXP1_VERSION",
    "Exp1Config",
    "run_exp1_profile_fold",
    "run_exp1",
]
missing_api = [name for name in required_exp1_api if not hasattr(exp1_rf, name)]
if missing_api:
    raise AttributeError(f"EXP-1 runner missing required API: {missing_api}")

print("EXP-1 runner:", exp1_rf.__file__)
print("EXP-1 version:", exp1_rf.EXP1_VERSION)
print("Static feature module:", static_features.__file__)
print("Static feature version:", static_features.STATIC_FEATURE_VERSION)
print("Static feature count:", len(static_features.FEATURE_COLUMNS))
print("Split manifest module:", split_manifest.__file__)
print("Evaluation module:", evaluation.__file__)

EXP-1 runner: /content/DiverseVul--IS-Project/vuln-detection/src/case_study_1/exp1/exp1_rf.py
EXP-1 version: cs1-exp1-svd-static-rf-v3-holdout-innercv-oob
Static feature module: /content/DiverseVul--IS-Project/vuln-detection/src/case_study_1/exp1/static_features.py
Static feature version: cs1-static-features-v1
Static feature count: 54
Split manifest module: /content/DiverseVul--IS-Project/vuln-detection/src/case_study_1/split_manifest.py
Evaluation module: /content/DiverseVul--IS-Project/vuln-detection/src/case_study_1/evaluation.py


In [ ]:
DRIVE_ROOT = Path(
    "/content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData"
)

PROCESSED_DIR = DRIVE_ROOT / "processed"
MANIFEST_ROOT = DRIVE_ROOT / "manifests"
OUTPUT_ROOT = DRIVE_ROOT / "outputs"

EXPERIMENT_ID = "cs1_project_holdout20_innercv_v1"
EXPERIMENT_MANIFEST_DIR = MANIFEST_ROOT / EXPERIMENT_ID
EXPERIMENT_OUTPUT_DIR = OUTPUT_ROOT / EXPERIMENT_ID

OUTER_SPLIT_DIR = EXPERIMENT_MANIFEST_DIR / "outer_holdout"
INNER_SPLIT_DIR = EXPERIMENT_MANIFEST_DIR / "inner_cv"

NORMALIZED_DATA_PATH = PROCESSED_DIR / "rdiversevul_cs1_normalized_v1.parquet"
ABSTRACTED_DATA_PATH = PROCESSED_DIR / "rdiversevul_cs1_normalized_plus_abstracted_v1.parquet"

OUTER_MANIFEST_PATH = OUTER_SPLIT_DIR / "cs1_outer_project_holdout_manifest.parquet"
INNER_MANIFEST_PATH = INNER_SPLIT_DIR / "cs1_project_grouped_5fold_manifest.parquet"
INNER_SELECTION_METADATA_PATH = INNER_SPLIT_DIR / "cs1_inner_grouped_split_selection_metadata.json"

STATIC_FEATURE_DIR = PROCESSED_DIR / "static_features"
STATIC_FEATURE_PATH = STATIC_FEATURE_DIR / "cs1_static_features_v1.parquet"

EXP1_IDABS_OUTPUT_DIR = EXPERIMENT_OUTPUT_DIR / f"exp1_rf_idabs_dev_cv_{RUN_VERSION}"
EXP1_IDABS_PROFILE_DIR = EXP1_IDABS_OUTPUT_DIR / "profile_fold"

for directory in [
    PROCESSED_DIR,
    MANIFEST_ROOT,
    OUTPUT_ROOT,
    EXPERIMENT_OUTPUT_DIR,
    STATIC_FEATURE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Abstracted dataset:", ABSTRACTED_DATA_PATH)
print("Outer manifest:", OUTER_MANIFEST_PATH)
print("Inner manifest:", INNER_MANIFEST_PATH)
print("Static-feature cache:", STATIC_FEATURE_PATH)
print("EXP-1-IDABS output:", EXP1_IDABS_OUTPUT_DIR)

Abstracted dataset: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/processed/rdiversevul_cs1_normalized_plus_abstracted_v1.parquet
Outer manifest: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/outer_holdout/cs1_outer_project_holdout_manifest.parquet
Inner manifest: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/inner_cv/cs1_project_grouped_5fold_manifest.parquet
Static-feature cache: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/processed/static_features/cs1_static_features_v1.parquet
EXP-1-IDABS output: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/outputs/cs1_project_holdout20_innercv_v1/exp1_rf_idabs_dev_cv_v1


## Dataset requirement

In [21]:
# ============================================================================
# 5. Load abstracted dataset and frozen manifests
# ============================================================================

if not ABSTRACTED_DATA_PATH.is_file():
    raise FileNotFoundError(
        "Missing abstracted dataset cache. Expected:\n"
        f"{ABSTRACTED_DATA_PATH}\n\n"
        "Run the identifier-abstraction build notebook first."
    )
if not OUTER_MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Missing frozen outer manifest:\n{OUTER_MANIFEST_PATH}")
if not INNER_MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Missing frozen inner manifest:\n{INNER_MANIFEST_PATH}")

full_df = pd.read_parquet(ABSTRACTED_DATA_PATH)
outer_manifest_df = pd.read_parquet(OUTER_MANIFEST_PATH)
inner_manifest_df = pd.read_parquet(INNER_MANIFEST_PATH)

required_dataset_columns = {
    "source_row_id",
    "normalized_code",
    "abstracted_code_v1",
    "label",
    "project",
}
missing_dataset_columns = required_dataset_columns.difference(full_df.columns)
if missing_dataset_columns:
    raise KeyError(
        "Abstracted dataset missing required columns:\n"
        f"{sorted(missing_dataset_columns)}\n\n"
        f"Available columns:\n{sorted(full_df.columns.tolist())}"
    )

required_outer_columns = {"source_row_id", "label", "project", "partition"}
missing_outer_columns = required_outer_columns.difference(outer_manifest_df.columns)
if missing_outer_columns:
    raise KeyError(f"Outer manifest missing columns: {sorted(missing_outer_columns)}")

required_inner_columns = {"source_row_id", "label", "project", "fold"}
missing_inner_columns = required_inner_columns.difference(inner_manifest_df.columns)
if missing_inner_columns:
    raise KeyError(f"Inner manifest missing columns: {sorted(missing_inner_columns)}")

print("Full abstracted dataset:", full_df.shape)
print("Columns:", sorted(full_df.columns.tolist()))
print("Outer manifest:", outer_manifest_df.shape)
print("Inner manifest:", inner_manifest_df.shape)
print("Outer partitions:")
print(outer_manifest_df["partition"].value_counts(dropna=False))

Full abstracted dataset: (261667, 6)
Columns: ['abstracted_code_v1', 'code', 'label', 'normalized_code', 'project', 'source_row_id']
Outer manifest: (261667, 5)
Inner manifest: (203958, 4)
Outer partitions:
partition
development      203958
outer_holdout     57709
Name: count, dtype: int64


In [ ]:
for name, frame in [
    ("full_df", full_df),
    ("outer_manifest_df", outer_manifest_df),
    ("inner_manifest_df", inner_manifest_df),
]:
    if frame["source_row_id"].duplicated().any():
        raise RuntimeError(f"{name} contains duplicate source_row_id values.")

full_ids = set(full_df["source_row_id"].tolist())
outer_ids = set(outer_manifest_df["source_row_id"].tolist())
inner_ids = set(inner_manifest_df["source_row_id"].tolist())

if full_ids != outer_ids:
    raise RuntimeError(
        "Outer manifest IDs do not exactly match dataset IDs.\n"
        f"Missing in manifest: {len(full_ids - outer_ids):,}\n"
        f"Extra in manifest: {len(outer_ids - full_ids):,}"
    )

observed_partitions = set(outer_manifest_df["partition"].dropna().astype(str))
expected_partitions = {"development", "outer_holdout"}
if observed_partitions != expected_partitions:
    raise RuntimeError(
        f"Unexpected outer partitions: {observed_partitions}. "
        f"Expected: {expected_partitions}"
    )


full_indexed = full_df.set_index("source_row_id", drop=False).sort_index()
outer_join = outer_manifest_df.set_index("source_row_id").join(
    full_indexed[["label", "project"]],
    rsuffix="_dataset",
)

if not (outer_join["label"].astype(int) == outer_join["label_dataset"].astype(int)).all():
    raise RuntimeError("Label mismatch between dataset and outer manifest.")
if not (
    outer_join["project"].astype(str).str.strip()
    == outer_join["project_dataset"].astype(str).str.strip()
).all():
    raise RuntimeError("Project mismatch between dataset and outer manifest.")

dev_ids = set(
    outer_manifest_df.loc[
        outer_manifest_df["partition"] == "development",
        "source_row_id",
    ].tolist()
)
holdout_ids = set(
    outer_manifest_df.loc[
        outer_manifest_df["partition"] == "outer_holdout",
        "source_row_id",
    ].tolist()
)

if inner_ids != dev_ids:
    raise RuntimeError(
        "Inner CV manifest must exactly equal development IDs.\n"
        f"Missing from inner manifest: {len(dev_ids - inner_ids):,}\n"
        f"Extra in inner manifest: {len(inner_ids - dev_ids):,}"
    )
if dev_ids.intersection(holdout_ids):
    raise RuntimeError("Development/holdout source-row overlap detected.")

dev_projects = set(
    outer_manifest_df.loc[
        outer_manifest_df["partition"] == "development",
        "project",
    ].astype(str).str.strip()
)
holdout_projects = set(
    outer_manifest_df.loc[
        outer_manifest_df["partition"] == "outer_holdout",
        "project",
    ].astype(str).str.strip()
)
project_overlap = dev_projects.intersection(holdout_projects)
if project_overlap:
    raise RuntimeError(
        "Development/outer-holdout project overlap detected. Examples: "
        f"{sorted(project_overlap)[:10]}"
    )


development_ids_sorted = sorted(dev_ids)
holdout_ids_sorted = sorted(holdout_ids)

dev_df = full_indexed.loc[development_ids_sorted].copy().reset_index(drop=True)
holdout_df = full_indexed.loc[holdout_ids_sorted].copy().reset_index(drop=True)

if set(dev_df["source_row_id"]).intersection(holdout_ids):
    raise RuntimeError("Holdout rows leaked into dev_df.")

split_summary = pd.DataFrame([
    {
        "partition": "development",
        "rows": len(dev_df),
        "projects": dev_df["project"].astype(str).nunique(),
        "vulnerable": int(dev_df["label"].astype(int).sum()),
        "positive_rate": float(dev_df["label"].astype(int).mean()),
    },
    {
        "partition": "outer_holdout_locked",
        "rows": len(holdout_df),
        "projects": holdout_df["project"].astype(str).nunique(),
        "vulnerable": int(holdout_df["label"].astype(int).sum()),
        "positive_rate": float(holdout_df["label"].astype(int).mean()),
    },
])
display(split_summary)

print("Validation complete: EXP-1-IDABS will use development partition only.")
print("Holdout rows are loaded only for split verification and remain locked.")

,partition,rows,projects,vulnerable,positive_rate
0,development,203958,594,10727,0.052594
1,outer_holdout_locked,57709,203,3211,0.055641


Validation complete: EXP-1-IDABS will use development partition only.
Holdout rows are loaded only for split verification and remain locked.


In [ ]:

selected_inner_split_seed = 42
if INNER_SELECTION_METADATA_PATH.exists():
    with INNER_SELECTION_METADATA_PATH.open("r", encoding="utf-8") as file:
        metadata = json.load(file)
    selected_inner_split_seed = int(metadata.get("selected_split_seed", 42))

INNER_N_SPLITS = 5
split_config = split_manifest.SplitConfig(
    n_splits=INNER_N_SPLITS,
    random_state=selected_inner_split_seed,
    shuffle=True,
)

split_manifest.assert_manifest_integrity(inner_manifest_df, config=split_config)

for fold_id in range(INNER_N_SPLITS):
    fold_test_projects = set(
        inner_manifest_df.loc[inner_manifest_df["fold"] == fold_id, "project"]
        .astype(str)
        .str.strip()
    )
    fold_train_projects = set(
        inner_manifest_df.loc[inner_manifest_df["fold"] != fold_id, "project"]
        .astype(str)
        .str.strip()
    )
    overlap = fold_test_projects.intersection(fold_train_projects)
    if overlap:
        raise RuntimeError(
            f"Project leakage inside inner fold {fold_id}: {sorted(overlap)[:10]}"
        )

fold_summary = (
    inner_manifest_df.groupby("fold")
    .agg(
        rows=("source_row_id", "count"),
        projects=("project", "nunique"),
        vulnerable=("label", "sum"),
    )
    .reset_index()
)
fold_summary["positive_rate"] = fold_summary["vulnerable"] / fold_summary["rows"]
display(fold_summary)
print("Inner manifest integrity passed.")

,fold,rows,projects,vulnerable,positive_rate
0,0,55509,1,2420,0.043597
1,1,40791,158,2091,0.051261
2,2,39072,146,2062,0.052774
3,3,35016,137,2047,0.058459
4,4,33570,152,2107,0.062764


Inner manifest integrity passed.


In [ ]:

EMPTY_ABSTRACTED_SENTINEL = "EMPTY_ABSTRACTED_CODE_SAMPLE"

for frame_name, frame in [("development", dev_df), ("holdout_locked", holdout_df)]:
    if CODE_COLUMN not in frame.columns:
        raise KeyError(
            f"{frame_name} frame is missing {CODE_COLUMN}. "
            f"Available columns: {sorted(frame.columns.tolist())}"
        )
    frame[CODE_COLUMN] = frame[CODE_COLUMN].fillna("").astype(str)
    empty_mask = frame[CODE_COLUMN].str.strip().eq("")
    n_empty = int(empty_mask.sum())
    print(f"Empty {CODE_COLUMN} rows in {frame_name}:", n_empty)
    if n_empty:
        frame.loc[empty_mask, CODE_COLUMN] = EMPTY_ABSTRACTED_SENTINEL
        print(
            f"Replaced {n_empty} empty rows in {frame_name} with "
            f"{EMPTY_ABSTRACTED_SENTINEL}."
        )

assert not dev_df[CODE_COLUMN].str.strip().eq("").any()
print("Non-empty abstracted-code guard passed for development data.")

Empty abstracted_code_v1 rows in development: 1
Replaced 1 empty rows in development with EMPTY_ABSTRACTED_CODE_SAMPLE.
Empty abstracted_code_v1 rows in holdout_locked: 0
Non-empty abstracted-code guard passed for development data.


## Static features

In [ ]:


if STATIC_FEATURE_PATH.exists() and not REBUILD_STATIC_FEATURE_CACHE:
    static_df = pd.read_parquet(STATIC_FEATURE_PATH)
    print("Loaded existing static-feature cache:", STATIC_FEATURE_PATH)
else:
    print("Static-feature cache missing or rebuild requested.")

    if "code" in full_df.columns:
        static_code_column = "code"
    else:
        static_code_column = "normalized_code"

    print("Static features will be extracted from:", static_code_column)

    static_config = static_features.StaticFeatureConfig(
        source_id_column="source_row_id",
        code_column=static_code_column,
        progress_every=25_000,
    )

    static_df = static_features.extract_static_feature_frame(
        full_df[["source_row_id", static_code_column]],
        config=static_config,
    )

    static_artifacts = static_features.save_static_feature_artifacts(
        static_frame=static_df,
        output_dir=STATIC_FEATURE_DIR,
        config=static_config,
        source_dataset_path=ABSTRACTED_DATA_PATH,
    )
    print("Static features saved:")
    for name, path in static_artifacts.items():
        print(f"  {name}: {path}")

required_static_columns = ["source_row_id"] + list(static_features.FEATURE_COLUMNS)
missing_static_columns = [c for c in required_static_columns if c not in static_df.columns]
if missing_static_columns:
    raise KeyError(f"Static feature cache missing columns: {missing_static_columns}")
if static_df["source_row_id"].duplicated().any():
    raise RuntimeError("Static feature cache contains duplicate source_row_id values.")

print("Static feature table shape:", static_df.shape)
display(static_df.head())

Loaded existing static-feature cache: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/processed/static_features/cs1_static_features_v1.parquet
Static feature table shape: (261667, 55)


,source_row_id,raw_char_count,raw_line_count,nonempty_line_count,avg_nonempty_line_length,max_line_length,comment_char_count,comment_line_count,comment_char_ratio,string_literal_count,...,memory_api_call_count,string_api_call_count,format_api_call_count,input_api_call_count,allocation_api_call_count,deallocation_api_call_count,unsafe_api_presence_count,sizeof_count,null_token_count,assert_call_count
0,0,6098.0,105.0,93.0,64.451613,151.0,52.0,1.0,0.008527,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,1732.0,65.0,55.0,30.327273,71.0,185.0,2.0,0.106813,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,115.0,4.0,4.0,28.000000,59.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,289.0,12.0,10.0,27.800000,50.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,824.0,27.0,23.0,34.695652,78.0,0.0,0.0,0.000000,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:


static_indexed = static_df.set_index("source_row_id", drop=False).sort_index()
dev_id_index = pd.Index(dev_df["source_row_id"].tolist())

missing_static_for_dev = dev_id_index.difference(static_indexed.index)
if len(missing_static_for_dev):
    raise RuntimeError(
        f"Missing static features for {len(missing_static_for_dev):,} development rows."
    )

dev_static_df = (
    static_indexed.loc[dev_id_index]
    .reset_index(drop=True)
    .copy()
)


if set(dev_static_df["source_row_id"]) != set(dev_df["source_row_id"]):
    raise RuntimeError("Development static features do not match dev_df IDs.")

print("Development frame:", dev_df.shape)
print("Development static features:", dev_static_df.shape)
print("Static features subset validation passed.")

Development frame: (203958, 6)
Development static features: (203958, 55)
Static features subset validation passed.


In [ ]:


exp1_config = exp1_rf.Exp1Config(
    experiment_name="cs1_exp1_rf_idabs_dev_grouped",
    code_column=CODE_COLUMN,
    source_id_column="source_row_id",
    label_column="label",
    project_column="project",
    fold_column="fold",
    n_splits=INNER_N_SPLITS,
    random_state=selected_inner_split_seed,
    decision_threshold=0.50,

  
    word_ngram_range=(1, 3),
    word_min_df=3,
    word_max_df=0.995,
    word_max_features=50_000,
    char_analyzer="char",
    char_ngram_range=(3, 4),
    char_min_df=8,
    char_max_df=0.995,
    char_max_features=60_000,
    lowercase=False,
    sublinear_tf=True,
    tfidf_norm="l2",

    svd_n_components=256,
    svd_algorithm="randomized",
    svd_n_iter=5,
    svd_n_oversamples=10,
    rf_n_estimators=200,
    rf_criterion="gini",
    rf_max_depth=None,
    rf_min_samples_split=2,
    rf_min_samples_leaf=2,
    rf_max_features="sqrt",
    rf_bootstrap=True,
    rf_max_samples=0.70,
    rf_class_weight="balanced_subsample",
    rf_n_jobs=-1,
    rf_oob_score=True,

   
    oob_threshold_min=0.005,
    oob_threshold_max=0.250,
    oob_threshold_step=0.005,
    oob_threshold_objective="f1",
    feature_importance_top_n=50,
    verbose=True,
)

print("EXP-1-IDABS configuration")
print("Experiment name:", exp1_config.experiment_name)
print("Input code column:", exp1_config.code_column)
print("Word max features:", exp1_config.word_max_features)
print("Char max features:", exp1_config.char_max_features)
print("SVD components:", exp1_config.svd_n_components)
print("RF trees:", exp1_config.rf_n_estimators)
print("Output directory:", EXP1_IDABS_OUTPUT_DIR)

EXP-1-IDABS configuration
Experiment name: cs1_exp1_rf_idabs_dev_grouped
Input code column: abstracted_code_v1
Word max features: 50000
Char max features: 60000
SVD components: 256
RF trees: 200
Output directory: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/outputs/cs1_project_holdout20_innercv_v1/exp1_rf_idabs_dev_cv_v1


In [ ]:

if RUN_PROFILE_FOLD:
    profile_start = time.perf_counter()
    profile_results = exp1_rf.run_exp1_profile_fold(
        normalized_frame=dev_df,
        static_features_frame=dev_static_df,
        manifest=inner_manifest_df,
        fold_id=PROFILE_FOLD_ID,
        config=exp1_config,
    )
    profile_seconds = time.perf_counter() - profile_start

    print("\nProfile run completed in %.2f minutes." % (profile_seconds / 60.0))
    print("\nOOB-threshold profile metrics:")
    print(json.dumps(profile_results["profile_metrics"], indent=2))

    print("\nFixed 0.50 threshold profile metrics:")
    print(json.dumps(profile_results["default_threshold_metrics"], indent=2))

    EXP1_IDABS_PROFILE_DIR.mkdir(parents=True, exist_ok=True)
    profile_results["predictions"].to_csv(
        EXP1_IDABS_PROFILE_DIR / "profile_fold_predictions.csv",
        index=False,
    )
    profile_results["feature_importances"].to_csv(
        EXP1_IDABS_PROFILE_DIR / "profile_fold_feature_importances.csv",
        index=False,
    )
    profile_results["training_metadata"].to_csv(
        EXP1_IDABS_PROFILE_DIR / "profile_fold_training_metadata.csv",
        index=False,
    )
    with (EXP1_IDABS_PROFILE_DIR / "profile_metrics.json").open("w", encoding="utf-8") as f:
        json.dump(profile_results["profile_metrics"], f, indent=2)

    print("Profile artifacts saved to:", EXP1_IDABS_PROFILE_DIR)
else:
    profile_results = None
    print("RUN_PROFILE_FOLD=False; profile skipped.")

RUN_PROFILE_FOLD=False; profile skipped.


In [ ]:


if RUN_OFFICIAL_DEV_CV:
    if EXP1_IDABS_OUTPUT_DIR.exists():
        existing_files = [p for p in EXP1_IDABS_OUTPUT_DIR.iterdir() if p.name != "profile_fold"]
        if existing_files and not ALLOW_OVERWRITE_OUTPUT_DIR:
            raise RuntimeError(
                "EXP-1-IDABS output directory already contains files.\n"
                f"Directory: {EXP1_IDABS_OUTPUT_DIR}\n\n"
                "To avoid mixing reruns, either:\n"
                "  1. Change RUN_VERSION to a new value, or\n"
                "  2. Set ALLOW_OVERWRITE_OUTPUT_DIR=True only if you deliberately want to replace it."
            )
        if existing_files and ALLOW_OVERWRITE_OUTPUT_DIR:
            print("Removing existing output directory because ALLOW_OVERWRITE_OUTPUT_DIR=True")
            shutil.rmtree(EXP1_IDABS_OUTPUT_DIR)

    EXP1_IDABS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    repo_commit = subprocess.check_output(
        ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"]
    ).decode().strip()

    official_start = time.perf_counter()
    exp1_idabs_results = exp1_rf.run_exp1(
        normalized_frame=dev_df,
        static_features_frame=dev_static_df,
        manifest=inner_manifest_df,
        config=exp1_config,
        output_dir=EXP1_IDABS_OUTPUT_DIR,
        additional_metadata={
            "run_kind": "exp1_idabs_development_only_cv",
            "representation_mode": REPRESENTATION_MODE,
            "input_column": CODE_COLUMN,
            "input_dataset_path": str(ABSTRACTED_DATA_PATH),
            "outer_manifest_path": str(OUTER_MANIFEST_PATH),
            "inner_manifest_path": str(INNER_MANIFEST_PATH),
            "static_feature_cache_path": str(STATIC_FEATURE_PATH),
            "repo_commit": repo_commit,
            "outer_holdout_used": False,
            "empty_abstracted_code_repair_token": EMPTY_ABSTRACTED_SENTINEL,
            "methodology_note": (
                "Supplementary EXP-1 identifier-abstraction experiment. "
                "Only the 80% development partition is used. The frozen 20% "
                "outer holdout is not scored here."
            ),
        },
    )
    official_seconds = time.perf_counter() - official_start

    print("\nOfficial EXP-1-IDABS development CV completed in %.2f minutes." % (official_seconds / 60.0))
    print("\nFixed 0.50-threshold pooled metrics:")
    print(evaluation.format_metric_report(exp1_idabs_results["evaluation"]["pooled_metrics"]))

    print("\nOOB-threshold pooled metrics:")
    print(json.dumps(exp1_idabs_results["oob_operating_evaluation"]["pooled_metrics"], indent=2))

    print("Artifacts saved to:", EXP1_IDABS_OUTPUT_DIR)
else:
    exp1_idabs_results = None
    print("RUN_OFFICIAL_DEV_CV=False; official five-fold CV skipped.")
    print("After profile succeeds, set RUN_OFFICIAL_DEV_CV=True and rerun this cell.")

[08:35:35] CS1-EXP1 official run started: 5-fold grouped CV.
[08:35:35] Configuration: lexical<= 110,000, SVD=256, static=54, RF trees=200.
[08:35:38] Fold 1/5 started | train=148,449, test=55,509, train projects=593, test projects=1.
[08:35:38] Fold 1/5 | fitting word TF-IDF...
[08:36:38] Fold 1/5 | word TF-IDF done in 59.4s (50,000 features).
[08:36:38] Fold 1/5 | fitting character TF-IDF...
[08:39:02] Fold 1/5 | character TF-IDF done in 2.40 min (60,000 features).
[08:39:02] Fold 1/5 | joining sparse lexical matrices...
[08:39:03] Fold 1/5 | sparse lexical matrix ready in 1.3s (110,000 columns).
[08:39:03] Fold 1/5 | fitting train-only TruncatedSVD (256 components)...
[08:42:18] Fold 1/5 | TruncatedSVD done in 3.25 min (explained variance ratio sum=0.3488).
[08:42:18] Fold 1/5 | loading cached static features...
[08:42:18] Fold 1/5 | combining SVD and static features...
[08:42:18] Fold 1/5 | training Random Forest (trees=200, max_features=sqrt, class_weight=balanced_subsample)...
[0

In [ ]:

safe_name = exp1_config.experiment_name.strip().lower().replace(" ", "_")

candidate_metric_paths = sorted(EXP1_IDABS_OUTPUT_DIR.glob(f"{safe_name}*pooled_metrics.json"))
candidate_fold_paths = sorted(EXP1_IDABS_OUTPUT_DIR.glob(f"{safe_name}*fold_metrics.csv"))
candidate_oob_metric_paths = sorted(EXP1_IDABS_OUTPUT_DIR.glob(f"{safe_name}*oob_operating_pooled_metrics.json"))
candidate_oob_fold_paths = sorted(EXP1_IDABS_OUTPUT_DIR.glob(f"{safe_name}*oob_operating_fold_metrics.csv"))

print("Output directory:", EXP1_IDABS_OUTPUT_DIR)
print("Fixed-threshold pooled metric files:", [p.name for p in candidate_metric_paths])
print("OOB-threshold pooled metric files:", [p.name for p in candidate_oob_metric_paths])

if candidate_metric_paths:
    with candidate_metric_paths[0].open("r", encoding="utf-8") as f:
        fixed_pooled_metrics = json.load(f)
    print("\nFixed 0.50-threshold pooled metrics:")
    print(json.dumps(fixed_pooled_metrics, indent=2))
else:
    fixed_pooled_metrics = None
    print("No fixed-threshold pooled metrics found yet.")

if candidate_oob_metric_paths:
    with candidate_oob_metric_paths[0].open("r", encoding="utf-8") as f:
        oob_pooled_metrics = json.load(f)
    print("\nOOB-threshold pooled metrics:")
    print(json.dumps(oob_pooled_metrics, indent=2))
else:
    oob_pooled_metrics = None
    print("No OOB-threshold pooled metrics found yet.")

if candidate_fold_paths:
    fixed_fold_metrics = pd.read_csv(candidate_fold_paths[0])
    print("\nFixed-threshold fold metrics:")
    display(fixed_fold_metrics)
if candidate_oob_fold_paths:
    oob_fold_metrics = pd.read_csv(candidate_oob_fold_paths[0])
    print("\nOOB-threshold fold metrics:")
    display(oob_fold_metrics)

Output directory: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/outputs/cs1_project_holdout20_innercv_v1/exp1_rf_idabs_dev_cv_v1
Fixed-threshold pooled metric files: ['cs1_exp1_rf_idabs_dev_grouped_oob_operating_pooled_metrics.json', 'cs1_exp1_rf_idabs_dev_grouped_pooled_metrics.json']
OOB-threshold pooled metric files: ['cs1_exp1_rf_idabs_dev_grouped_oob_operating_pooled_metrics.json']

Fixed 0.50-threshold pooled metrics:
{
  "n_samples": 203958,
  "vulnerable_1": 10727,
  "non_vulnerable_0": 193231,
  "positive_rate": 0.05259416154306279,
  "threshold": null,
  "average_precision_pr_auc": 0.12401256032777705,
  "precision": 0.15755063919210788,
  "recall": 0.25160809173114573,
  "f1": 0.1937683968698399,
  "mcc": 0.142378360739403,
  "specificity": 0.9253121911080521,
  "negative_predictive_value": 0.9570297655049859,
  "false_positive_rate": 0.07468780889194798,
  "false_negative_rate": 0.7483919082688543,
  "true_negative": 178799,
  "false_positive": 

,fold,n_samples,vulnerable_1,non_vulnerable_0,positive_rate,threshold,average_precision_pr_auc,precision,recall,f1,...,negative_predictive_value,false_positive_rate,false_negative_rate,true_negative,false_positive,false_negative,true_positive,predicted_positive,predicted_positive_rate,test_unique_projects
0,0,55509,2420,53089,0.043597,0.5,0.084580,0.000000,0.000000,0.000000,...,0.956400,0.000075,1.000000,53085,4,2420,0,4,0.000072,1
1,1,40791,2091,38700,0.051261,0.5,0.113763,0.333333,0.002391,0.004748,...,0.948842,0.000258,0.997609,38690,10,2086,5,15,0.000368,158
2,2,39072,2062,37010,0.052774,0.5,0.125221,0.090909,0.000485,0.000965,...,0.947236,0.000270,0.999515,37000,10,2061,1,11,0.000282,146
3,3,35016,2047,32969,0.058459,0.5,0.153411,0.000000,0.000000,0.000000,...,0.941536,0.000091,1.000000,32966,3,2047,0,3,0.000086,137
4,4,33570,2107,31463,0.062764,0.5,0.154290,0.263158,0.002373,0.004704,...,0.937349,0.000445,0.997627,31449,14,2102,5,19,0.000566,152



OOB-threshold fold metrics:


,n_samples,vulnerable_1,non_vulnerable_0,positive_rate,threshold,average_precision_pr_auc,precision,recall,f1,mcc,...,false_positive_rate,false_negative_rate,true_negative,false_positive,false_negative,true_positive,predicted_positive,predicted_positive_rate,fold,test_unique_projects
0,55509,2420,53089,0.043597,0.125,0.084580,0.118453,0.142975,0.129564,0.086398,...,0.048503,0.857025,50514,2575,2074,346,2921,0.052622,0,1
1,40791,2091,38700,0.051261,0.140,0.113763,0.144753,0.219034,0.174310,0.122935,...,0.069922,0.780966,35994,2706,1633,458,3164,0.077566,1,158
2,39072,2062,37010,0.052774,0.120,0.125221,0.153592,0.293404,0.201633,0.150985,...,0.090084,0.706596,33676,3334,1457,605,3939,0.100814,2,146
3,35016,2047,32969,0.058459,0.130,0.153411,0.176370,0.301905,0.222663,0.167592,...,0.087537,0.698095,30083,2886,1429,618,3504,0.100069,3,137
4,33570,2107,31463,0.062764,0.125,0.154290,0.186511,0.318937,0.235377,0.176915,...,0.093157,0.681063,28532,2931,1435,672,3603,0.107328,4,152


## 14. Confidence interval on development-CV pooled PR-AUC

In [ ]:
import case_study_1.confidence_intervals as confidence_intervals

exp1_idabs_devcv_ci = confidence_intervals.bootstrap_metric_ci(
    exp1_idabs_results["oof_predictions"],
    metric="average_precision_pr_auc",
    n_bootstrap=1000,
    random_state=42,
)
print(confidence_intervals.format_ci_report(exp1_idabs_devcv_ci))

In [ ]:


comparison_rows = [
    {
        "experiment": "EXP-0 fixed normalized_code",
        "representation": "normalized_code",
        "scope": "development pooled OOF",
        "pr_auc": 0.125205,
        "status": "completed reference",
    },
    {
        "experiment": "EXP-0 nested-alpha normalized_code",
        "representation": "normalized_code",
        "scope": "development nested pooled OOF",
        "pr_auc": 0.141971,
        "status": "completed reference",
    },
    {
        "experiment": "EXP-0-IDABS nested-alpha",
        "representation": "abstracted_code_v1",
        "scope": "development nested pooled OOF",
        "pr_auc": 0.136888,
        "status": "completed reference",
    },
    {
        "experiment": "EXP-1 RF normalized_code",
        "representation": "normalized_code + static features",
        "scope": "development pooled OOF",
        "pr_auc": 0.124803,
        "status": "completed reference",
    },
]

if fixed_pooled_metrics is not None:
    comparison_rows.append(
        {
            "experiment": "EXP-1-IDABS RF",
            "representation": "abstracted_code_v1 + static features",
            "scope": "development pooled OOF",
            "pr_auc": float(fixed_pooled_metrics["average_precision_pr_auc"]),
            "status": "this notebook, fixed 0.50 diagnostic",
        }
    )

comparison_df = pd.DataFrame(comparison_rows).sort_values(
    "pr_auc",
    ascending=False,
).reset_index(drop=True)

display(comparison_df)

,experiment,representation,scope,pr_auc,status
0,EXP-0 nested-alpha normalized_code,normalized_code,development nested pooled OOF,0.141971,completed reference
1,EXP-0-IDABS nested-alpha,abstracted_code_v1,development nested pooled OOF,0.136888,completed reference
2,EXP-0 fixed normalized_code,normalized_code,development pooled OOF,0.125205,completed reference
3,EXP-1 RF normalized_code,normalized_code + static features,development pooled OOF,0.124803,completed reference
4,EXP-1-IDABS RF,abstracted_code_v1 + static features,development pooled OOF,0.124013,"this notebook, fixed 0.50 diagnostic"


## Final outer-holdout evaluation

In [ ]:
if RUN_FINAL_OUTER_HOLDOUT:
    FINAL_HOLDOUT_OUTPUT_DIR = EXPERIMENT_OUTPUT_DIR / f"exp1_rf_idabs_final_holdout_{RUN_VERSION}"
    FINAL_HOLDOUT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    exp1_idabs_final_holdout = exp1_rf.run_exp1_final_holdout(
        dev_frame=dev_df,
        static_features_frame=static_df,
        holdout_frame=holdout_df,
        config=exp1_config,
        output_dir=FINAL_HOLDOUT_OUTPUT_DIR,
        additional_metadata={
            "run_kind": "exp1_idabs_final_outer_holdout",
            "representation_mode": REPRESENTATION_MODE,
            "input_column": CODE_COLUMN,
            "selection_source": "development_pooled_oof_pr_auc",
        },
    )

    print(evaluation.format_metric_report(exp1_idabs_final_holdout["metrics"]))
else:
    exp1_idabs_final_holdout = None
    print("Final outer holdout remains locked. EXP-1-IDABS is development-only.")

## Holdout precision-recall curve and confusion matrix

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(exp1_idabs_final_holdout["artifact_paths"]["pr_curve"])))
display(Image(filename=str(exp1_idabs_final_holdout["artifact_paths"]["confusion_matrix"])))

## Holdout feature importances

In [ ]:
display(exp1_idabs_final_holdout["feature_importances"].head(30))

## Holdout false positives and false negatives

In [ ]:
holdout_errors = exp1_idabs_final_holdout["predictions"].merge(
    holdout_df[["source_row_id", CODE_COLUMN]],
    on="source_row_id",
    how="left",
)

false_positives = holdout_errors.loc[(holdout_errors["label"] == 0) & (holdout_errors["y_pred"] == 1)]
false_negatives = holdout_errors.loc[(holdout_errors["label"] == 1) & (holdout_errors["y_pred"] == 0)]

print(len(false_positives), len(false_negatives))
display(false_positives.sort_values("y_score", ascending=False).head(5))
display(false_negatives.sort_values("y_score", ascending=True).head(5))

## Final holdout report summary

In [ ]:
holdout_report_summary = pd.DataFrame([{
    "experiment": "CS1-EXP1-IDABS-RF",
    "representation": CODE_COLUMN,
    "dev_samples": len(dev_df),
    "holdout_samples": len(holdout_df),
    "pr_auc": exp1_idabs_final_holdout["metrics"]["average_precision_pr_auc"],
    "precision": exp1_idabs_final_holdout["metrics"]["precision"],
    "recall": exp1_idabs_final_holdout["metrics"]["recall"],
    "f1": exp1_idabs_final_holdout["metrics"]["f1"],
    "mcc": exp1_idabs_final_holdout["metrics"]["mcc"],
    "threshold": exp1_idabs_final_holdout["metrics"]["threshold"],
}])

holdout_report_summary.to_csv(
    FINAL_HOLDOUT_OUTPUT_DIR / "cs1_exp1_idabs_holdout_report_summary.csv",
    index=False,
)
display(holdout_report_summary)